# Мощность и снижение дисперсии в VarWar

Расчёт по сохранённым агрегатам синтетических рандомизаций VarWar. Сценарий `contaminated`, полная рандомизация, эффект 0,1, только предпилотные признаки. Основная серия v5 — 200 повторов на эффект, подтверждающая v3 — 1 000. В [выгрузке](../../results/11_varwar/aggregates.csv) сохранены строки пяти процедур основной серии и двух процедур подтверждающей. Исследовательский движок — [VarWar](https://github.com/Yurashku/VarWar).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

relative = Path("results/11_varwar/aggregates.csv")
source = next(parent / relative for parent in (Path.cwd(), *Path.cwd().parents)
              if (parent / relative).is_file())
rows = pd.read_csv(source)
for series, frame in rows.groupby("series", sort=False):
    baseline = frame.loc[frame.method == "Уэлч", "empirical_sd"].item()
    reduction = 1 - (frame.empirical_sd / baseline) ** 2
    assert np.allclose(reduction, frame.variance_reduction, atol=1e-7)
    summary = frame[["method", "repeats"]].copy()
    summary["снижение дисперсии, %"] = 100 * reduction
    summary["мощность, %"] = 100 * frame.power
    summary["ложные срабатывания, %"] = 100 * frame.false_positive_rate
    summary["покрытие, %"] = 100 * frame.coverage
    print(series)
    print(summary.round(1).to_string(index=False))

combined
     method  repeats  снижение дисперсии, %  мощность, %  ложные срабатывания, %  покрытие, %
       Уэлч      200                    0.0         59.0                     4.0         96.0
CUPED HypEx      200                   16.3         68.5                     3.5         96.5
CUPAC HypEx      200                   30.4         77.0                     4.5         95.5
  Лин (HC3)      200                   39.7         84.5                     3.0         97.0
    ML-RATE      200                   39.6         85.0                     3.0         97.0
confirmation
   method  repeats  снижение дисперсии, %  мощность, %  ложные срабатывания, %  покрытие, %
     Уэлч     1000                    0.0         61.0                     5.1         94.9
Лин (HC3)     1000                   39.4         80.6                     4.8         95.2


In [2]:
main = rows[rows.series == "combined"].set_index("method")
gap = main.loc["ML-RATE", "power"] - main.loc["Лин (HC3)", "power"]
print(f"Разница между лидерами: {gap * 100:.1f} п.п., "
      f"{round(gap * main.loc['ML-RATE', 'repeats'])} обнаружение на серию.")
confirm = rows[rows.series == "confirmation"].set_index("method")
print("Подтверждающая серия, ложные срабатывания Лина:",
      round(confirm.loc["Лин (HC3)", "false_positive_rate"] * confirm.loc["Лин (HC3)", "repeats"]))

Разница между лидерами: 0.5 п.п., 1 обнаружение на серию.
Подтверждающая серия, ложные срабатывания Лина: 48


## Граница вывода

Близкая мощность в этом сценарии послужила основанием рассмотреть Лина для HypEx. В конкретном пилоте ожидаемый выигрыш и размер выборки следует оценивать по его метрике и предпилотным данным.